# FinSight AI · Research Part 2: Agents and memory

Part 1 settled the building blocks: bge-m3 embeddings for routing and search, and "compute numbers with code, mark them verified when two routes agree". Part 2 asks how to **put the blocks together**, and whether the multi-agent design FinSight plans is actually better than a single agent. Every experiment uses **real public data**, with no curated or synthetic examples.

| Exp | Question | Real dataset | Arms |
|---|---|---|---|
| **E4** | Does a router plus specialist agents beat a single agent? | **TAT-QA**: 1,668 dev questions over real annual-report tables and text, with four answer types (span, multi-span, arithmetic, count) | Single LLM; single agent with a calculator tool; learned router + specialists; LLM orchestrator + specialists; router + specialists + checks; oracle router (ceiling) |
| **E5** | Which memory design keeps a multi-turn conversation correct? | **ConvFinQA**: 421 dev conversations (1,490 turns) over real S&P 500 earnings reports, where later questions refer back to earlier answers | No memory; full transcript; structured memory (named results the code can reuse) |

Each LLM experiment runs on **two backends with identical prompts**: a local open model (Qwen 2.5 7B in 4-bit, reproducible) and, if you add an API key, a free-tier API model. The paper claim is that the method helps both.

**How to run on Kaggle**
1. *Settings → Accelerator → GPU T4 x2*, and *Settings → Internet → On*.
2. Optional API backend: *Add-ons → Secrets*, add a secret named `LLM_API_KEY` and tick it for this notebook. The default provider is Google Gemini (free key from aistudio.google.com); any OpenAI-compatible provider works by changing `API_BASE_URL` and `API_MODEL`.
3. *Save Version → Save & Run All (Commit)*. Set `FAST = True` first for a 15-minute smoke test.

Results are written to `/kaggle/working/part2_results/`. Every generation is cached, so a re-run (for example after an API rate limit) continues where it stopped.

## 0. Configuration

In [ ]:
FAST = False          # True: small samples for a quick end-to-end check
SEED = 13

RUN_E4_AGENTS = True
RUN_E5_MEMORY = True

# Local backend (always runs when a GPU is present)
LLM_NAME     = "Qwen/Qwen2.5-7B-Instruct"   # loaded in 4-bit; falls back to FALLBACK_LLM if 4-bit loading fails
LLM_4BIT     = True
FALLBACK_LLM = "Qwen/Qwen2.5-3B-Instruct"
ROUTER_EMBED = "BAAI/bge-m3"                # Part 1's best embedding model

# API backend (runs only if a key is found in Kaggle Secrets under API_SECRET_NAME)
USE_API         = True
API_SECRET_NAME = "LLM_API_KEY"
API_BASE_URL    = "https://generativelanguage.googleapis.com/v1beta/openai/"   # Groq: https://api.groq.com/openai/v1/
API_MODEL       = "gemini-2.5-flash-lite"                                        # Groq: llama-3.3-70b-versatile
API_RPM         = 10      # requests per minute; keep under your free-tier limit

# Sample sizes (the API backend runs on the first API_* items of the same random sample, so results pair up)
TATQA_N     = 60 if FAST else 300    # TAT-QA dev questions (max 1668)
CONV_N      = 15 if FAST else 100    # ConvFinQA dev conversations (max 421)
API_TATQA_N = 20 if FAST else 100
API_CONV_N  = 5 if FAST else 30

AGENT_MAX_STEPS = 3    # calculator calls the single agent may make before it must answer
GEN_BATCH       = 16

## 1. Setup

In [ ]:
import subprocess, sys, importlib

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=False)

for pkg, pip_name in [("sentence_transformers", "sentence-transformers"), ("transformers", None),
                      ("sklearn", "scikit-learn"), ("scipy", None), ("pandas", None), ("matplotlib", None),
                      ("requests", None)]:
    ensure(pkg, pip_name)
if LLM_4BIT:
    ensure("bitsandbytes")

In [ ]:
import os, re, json, math, time, random, zipfile, hashlib, signal, urllib.request, platform
from collections import Counter, defaultdict
from decimal import Decimal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

random.seed(SEED); np.random.seed(SEED)

OUT = "/kaggle/working/part2_results" if os.path.isdir("/kaggle/working") else os.path.abspath("part2_results")
CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
except ImportError:
    torch, DEVICE = None, "cpu"

RESULTS = {"config": dict(FAST=FAST, SEED=SEED, LLM_NAME=LLM_NAME, LLM_4BIT=LLM_4BIT, ROUTER_EMBED=ROUTER_EMBED,
                          API_BASE_URL=API_BASE_URL, API_MODEL=API_MODEL, TATQA_N=TATQA_N, CONV_N=CONV_N,
                          API_TATQA_N=API_TATQA_N, API_CONV_N=API_CONV_N, AGENT_MAX_STEPS=AGENT_MAX_STEPS),
           "env": dict(python=platform.python_version(), device=DEVICE,
                       gpu=(torch.cuda.get_device_name(0) if torch is not None and DEVICE == "cuda" else None),
                       torch=getattr(torch, "__version__", None)),
           "E4": {}, "E5": {}}
SUMMARY_ROWS = []
print(RESULTS["env"], "| output:", OUT)

def download(url, name):
    path = os.path.join(CACHE, name)
    if not os.path.exists(path):
        print("downloading", url)
        req = urllib.request.Request(url, headers={"User-Agent": "finsight-research"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def bootstrap_ci(values, n=1000, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n, len(values)))].mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def paired_diff_ci(a, b, n=1000, seed=SEED):
    """Mean of a - b over the same questions, with a paired bootstrap 95% CI."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    lo, hi = bootstrap_ci(d, n, seed)
    return float(d.mean()), lo, hi

def add_summary(exp, backend, method, metric, value, ci=None, **extra):
    row = dict(experiment=exp, backend=backend, method=method, metric=metric, value=round(float(value), 4))
    if ci is not None:
        row["ci_low"], row["ci_high"] = round(ci[0], 4), round(ci[1], 4)
    row.update({k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in extra.items()})
    SUMMARY_ROWS.append(row)

def save_all():
    pd.DataFrame(SUMMARY_ROWS).to_csv(os.path.join(OUT, "summary.csv"), index=False)
    with open(os.path.join(OUT, "results.json"), "w") as f:
        json.dump(RESULTS, f, indent=2, default=str)

## 2. Shared tools from Part 1

The sandboxed program runner, number parsing and answer matching are the same as in Part 1, so the two parts' numbers are comparable. The runner now also accepts named variables, which the structured-memory arm in E5 uses.

In [ ]:
NUM_RE = re.compile(r"\(?-?\$?\d[\d,]*\.?\d*%?\)?")

def parse_number(tok):
    s = str(tok).strip()
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()").replace("$", "").replace(",", "").rstrip("%").strip()
    if s in {"", "-", "."}:
        return None
    try:
        v = float(s)
    except ValueError:
        return None
    return -v if neg else v

def table_to_text(table):
    return "\n".join(" | ".join(str(c).strip() for c in row) for row in table)

class _Timeout(Exception):
    pass

def run_program(code, seconds=3, variables=None):
    safe_builtins = {"abs": abs, "min": min, "max": max, "sum": sum, "round": round, "len": len,
                     "float": float, "int": int, "range": range, "pow": pow, "sorted": sorted,
                     "bool": bool, "list": list, "dict": dict, "zip": zip, "enumerate": enumerate}
    env = {"__builtins__": safe_builtins, "math": math}
    env.update(variables or {})
    if re.search(r"\b(import|open|exec|eval|__|os\.|sys\.)", code.replace("import math", "")):
        return None, "blocked: program uses a forbidden construct"
    def handler(signum, frame):
        raise _Timeout()
    old = signal.signal(signal.SIGALRM, handler) if hasattr(signal, "SIGALRM") else None
    try:
        if old is not None:
            signal.alarm(seconds)
        exec(code.replace("import math", ""), env)
        if "ans" not in env:
            return None, "the program did not set a variable named ans"
        return env["ans"], None
    except _Timeout:
        return None, "the program timed out"
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    finally:
        if old is not None:
            signal.alarm(0); signal.signal(signal.SIGALRM, old)

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, flags=re.S)
    if m:
        return m.group(1).strip()
    m = re.search(r"```(?:python)?\s*\n(.*)", text, flags=re.S)   # unclosed block (output cut off)
    return (m.group(1) if m else text).strip()

def normalise_pred(v):
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"yes", "no", "true", "false"}:
            return {"true": "yes", "false": "no"}.get(s, s)
        return parse_number(s)
    if isinstance(v, (int, float, Decimal, np.floating, np.integer)):
        f = float(v)
        return f if math.isfinite(f) else None
    return None

def close(a, b, rel=1e-2, abs_tol=5e-3):
    return abs(a - b) <= max(rel * abs(b), abs_tol)

def answers_agree(a, b):
    a, b = normalise_pred(a), normalise_pred(b)
    if a is None or b is None:
        return False
    if isinstance(a, str) or isinstance(b, str):
        return a == b
    return any(close(c, b) for c in (a, a * 100, a / 100))

def finqa_correct(pred, gold):
    """FinQA/ConvFinQA execution accuracy: 1% tolerance, percentage given as a fraction (or the reverse) accepted."""
    pred = normalise_pred(pred)
    if pred is None:
        return False
    if isinstance(gold, str):
        g = gold.strip().lower()
        if g in {"yes", "no"}:
            return pred == g
        gold = parse_number(g)
        if gold is None:
            return False
    if isinstance(pred, str):
        return False
    return any(close(c, float(gold)) for c in (pred, pred * 100, pred / 100))

PROGRAM_INSTR = ("First, in two or three short lines, say which numbers you need and the formula. "
                 "Then write a short Python program that computes the answer and stores it in a variable named ans. "
                 "Copy every number exactly as it appears in the report; do not invent or estimate values. "
                 "For yes/no questions set ans to True or False. Put the code in a ```python block.")

## 3. LLM backends

Both backends get the same system prompt and messages, use greedy decoding (temperature 0), and record prompt and output tokens per call so every arm's cost can be reported. Generations are stored in `cache/generations.jsonl` keyed by backend, model and prompt, so nothing is generated twice.

In [ ]:
SYSTEM = "You are a careful financial analyst. Use only the information in the provided report excerpt."

class APIStop(Exception):
    pass

class LocalBackend:
    name = "local"

    def __init__(self):
        self.tok = self.model = None
        self._model_id = None

    @property
    def model_id(self):
        if self._model_id is None:
            self.load()
        return self._model_id

    def load(self):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        def tokenizer(name):
            tok = AutoTokenizer.from_pretrained(name, padding_side="left", truncation_side="left")
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            return tok
        kwargs = dict(device_map={"": 0})   # one GPU: splitting the model over two T4s is slower
        if LLM_4BIT:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        else:
            kwargs["torch_dtype"] = torch.float16
        try:
            self.tok = tokenizer(LLM_NAME)
            self.model = AutoModelForCausalLM.from_pretrained(LLM_NAME, **kwargs).eval()
            self._model_id = LLM_NAME + (" (4-bit)" if LLM_4BIT else "")
        except Exception as exc:
            print(f"Could not load {LLM_NAME} ({type(exc).__name__}: {exc}); falling back to {FALLBACK_LLM} in fp16")
            self.tok = tokenizer(FALLBACK_LLM)
            self.model = AutoModelForCausalLM.from_pretrained(FALLBACK_LLM, device_map={"": 0},
                                                              torch_dtype=torch.float16).eval()
            self._model_id = FALLBACK_LLM
        print("Local LLM in use:", self._model_id)
        RESULTS["config"]["LOCAL_LLM_USED"] = self._model_id

    def run(self, convs, max_new_tokens):
        tok, model = self.tok, self.model
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}] + c, tokenize=False,
                                         add_generation_prompt=True) for c in convs]
        out = []
        for i in range(0, len(texts), GEN_BATCH):
            chunk = texts[i:i + GEN_BATCH]
            t0 = time.perf_counter()
            enc = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=6144).to(model.device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            new = gen[:, enc["input_ids"].shape[1]:]
            sec = (time.perf_counter() - t0) / len(chunk)
            decoded = tok.batch_decode(new, skip_special_tokens=True)
            for j, text in enumerate(decoded):
                out.append(dict(text=text, in_tok=int(enc["attention_mask"][j].sum()),
                                out_tok=int((new[j] != tok.pad_token_id).sum()), sec=sec))
        return out

class APIBackend:
    name = "api"

    def __init__(self, key):
        self.key, self.model_id = key, API_MODEL
        self.url = API_BASE_URL.rstrip("/") + "/chat/completions"
        self.last, self.failures = 0.0, 0

    def run(self, convs, max_new_tokens):
        return [self._one(c, max_new_tokens) for c in convs]

    def _one(self, msgs, max_new_tokens):
        body = {"model": API_MODEL, "temperature": 0, "max_tokens": max_new_tokens,
                "messages": [{"role": "system", "content": SYSTEM}] + msgs}
        for attempt in range(6):
            wait = self.last + 60.0 / API_RPM - time.time()
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            t0 = time.perf_counter()
            try:
                r = requests.post(self.url, json=body, timeout=120,
                                  headers={"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"})
            except requests.RequestException as exc:
                print("  API request failed:", type(exc).__name__)
                time.sleep(5 * 2 ** attempt)
                continue
            if r.status_code == 200:
                j = r.json()
                usage = j.get("usage") or {}
                self.failures = 0
                return dict(text=j["choices"][0]["message"].get("content") or "",
                            in_tok=usage.get("prompt_tokens", 0), out_tok=usage.get("completion_tokens", 0),
                            sec=time.perf_counter() - t0)
            if r.status_code in (429, 500, 502, 503, 504):
                ra = r.headers.get("retry-after", "")
                time.sleep(min(float(ra) if re.fullmatch(r"\d+(\.\d+)?", ra) else 5 * 2 ** attempt, 120))
                continue
            raise APIStop(f"API error {r.status_code}: {r.text[:300]}")
        self.failures += 1
        if self.failures >= 3:
            raise APIStop("The API keeps refusing (rate limit or daily quota). Re-run later: finished answers are cached.")
        return dict(text="", in_tok=0, out_tok=0, sec=0.0, error=True)

GEN_STORE_PATH = os.path.join(CACHE, "generations.jsonl")
_GEN_STORE = {}
if os.path.exists(GEN_STORE_PATH):
    with open(GEN_STORE_PATH) as f:
        for line in f:
            rec = json.loads(line)
            _GEN_STORE[rec["key"]] = rec

def llm(backend, tag, convs, max_new_tokens):
    """One generation record per conversation (list of chat messages). Cached and resumable."""
    keys = [hashlib.sha1(json.dumps([backend.name, backend.model_id, max_new_tokens, SYSTEM, c]).encode()).hexdigest()
            for c in convs]
    todo = {}
    for k, c in zip(keys, convs):
        if k not in _GEN_STORE:
            todo[k] = c
    print(f"[{backend.name}] {tag}: {len(convs) - len(todo)} cached, generating {len(todo)}", flush=True)
    order = sorted(todo, key=lambda k: len(json.dumps(todo[k])))   # similar lengths per batch: less padding
    step = GEN_BATCH * 4 if backend.name == "local" else 10
    t_start = time.perf_counter()
    for i in range(0, len(order), step):
        chunk = order[i:i + step]
        recs = backend.run([todo[k] for k in chunk], max_new_tokens)
        with open(GEN_STORE_PATH, "a") as f:
            for k, rec in zip(chunk, recs):
                if rec.get("error"):
                    continue
                rec["key"] = k
                _GEN_STORE[k] = rec
                f.write(json.dumps(rec) + "\n")
        print(f"  {min(i + step, len(order))}/{len(order)} ({time.perf_counter() - t_start:.0f}s)", end="\r", flush=True)
    if order:
        print()
    return [_GEN_STORE.get(k, dict(text="", in_tok=0, out_tok=0, sec=0.0, error=True)) for k in keys]

def get_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(API_SECRET_NAME)
    except Exception:
        return os.environ.get(API_SECRET_NAME)

BACKENDS = []
if torch is not None and DEVICE == "cuda":
    BACKENDS.append(LocalBackend())
else:
    print("No GPU found: the local backend is skipped (turn on GPU T4 x2 in Settings).")
_key = get_api_key() if USE_API else None
if _key:
    BACKENDS.append(APIBackend(_key))
    print(f"API backend on: {API_MODEL} at {API_BASE_URL}")
else:
    print(f"No secret named {API_SECRET_NAME}: the API backend is skipped.")

## E4 · Single agent versus router + specialists on TAT-QA

TAT-QA questions come from real annual reports. Each has a table and its paragraphs, and needs one of four kinds of answer: copy one span, copy several spans, calculate, or count. That mirrors FinSight, where some questions need a lookup and others need arithmetic.

**Arms** (identical context and answer rules in every arm):

| Arm | Design | LLM calls |
|---|---|---|
| **S1** · Single LLM | One call answers directly, after brief reasoning | 1 |
| **S2** · Single agent + calculator | One agent that can call a calculator tool up to 3 times, then answers (ReAct style). This is the strong single-agent baseline. | 1–4 |
| **M1** · Learned router + specialists | bge-m3 + logistic regression router, trained on TAT-QA *train* questions, sends each question to a specialist: **lookup** (copy spans), **calculator** (Part 1's reason-then-program) or **counter** (list items, count in code) | 1 |
| **M2** · LLM orchestrator + specialists | The LLM itself decides which specialist to call | 2 |
| **M3** · Router + specialists + checks | M1, but a calculated answer is kept only when it agrees with S1, and a copied span only when it appears verbatim in the report; otherwise abstain | 2 |
| **Oracle** · Gold route + specialists | Uses TAT-QA's gold answer type: the ceiling for any router | 1 |

Scoring uses the **official TAT-QA scorer** (`tatqa_metric.py`, downloaded from the TAT-QA repository): exact match and numeracy-focused F1, with the answer's scale (thousand, million, percent ...) required to match. Every wrong answer is also labelled with a failure mode (wrong route, wrong scale, sign error, percent vs fraction, program failed, wrong number, wrong span), which the paper needs for its error analysis.

Specialists are run once per question and the arms are composed from them, so the comparison is exact and no generation is wasted. Cost per arm counts only the calls that arm would make on its own.

In [ ]:
TATQA_RAW = "https://raw.githubusercontent.com/NExTplusplus/TAT-QA/master/"
GROUP = {"span": "lookup", "multi-span": "lookup", "arithmetic": "calculator", "count": "counter"}
SCALE_WORDS = {"none": "", "": "", "n/a": "", "unit": "", "units": "", "thousand": "thousand", "thousands": "thousand",
               "million": "million", "millions": "million", "billion": "billion", "billions": "billion",
               "percent": "percent", "percentage": "percent", "%": "percent"}

if RUN_E4_AGENTS:
    for f in ("tatqa_metric.py", "tatqa_utils.py"):
        download(TATQA_RAW + f, f)
    sys.path.insert(0, CACHE)
    from tatqa_metric import TaTQAEmAndF1
    from tatqa_utils import to_number, normalize_answer

    tat_dev = json.load(open(download(TATQA_RAW + "dataset_raw/tatqa_dataset_dev.json", "tatqa_dev.json")))
    tat_train = json.load(open(download(TATQA_RAW + "dataset_raw/tatqa_dataset_train.json", "tatqa_train.json")))
    tat_all = [dict(doc=d, q=q) for d in tat_dev for q in d["questions"]]
    tat_items = random.Random(SEED).sample(tat_all, min(TATQA_N, len(tat_all)))
    train_q = [q for d in tat_train for q in d["questions"]]
    print(f"TAT-QA: {len(tat_items)} of {len(tat_all)} dev questions; router trained on {len(train_q)} train questions")
    print("answer types in sample:", Counter(it["q"]["answer_type"] for it in tat_items))

def tat_context(doc):
    paras = "\n".join(p["text"] for p in sorted(doc["paragraphs"], key=lambda p: p["order"]))
    return f"Table:\n{table_to_text(doc['table']['table'])}\n\nText:\n{paras}"

ANSWER_RULES = ("Answer rules: for text answers copy the exact words from the report; if there are several items, "
                "separate them with ' | '. For numbers write only the number (no currency sign, commas or % sign) "
                "in the report's own units: if the report gives amounts in thousands or millions, use the number as "
                "written and set the scale. Write percentages as percent numbers, for example 12.5 for 12.5%.")
SCALE_LINE = "Scale: <none, thousand, million, billion or percent>"

def tat_header(it):
    return f"Report:\n{tat_context(it['doc'])}\n\nQuestion: {it['q']['question']}\n\n"

def prompt_s1(it):
    return tat_header(it) + ANSWER_RULES + f"\n\nThink briefly, then end with exactly two lines:\nAnswer: <answer>\n{SCALE_LINE}"

def prompt_agent(it):
    return (tat_header(it) + ANSWER_RULES + "\n\nYou have a calculator. To use it, write one line "
            "`CALC: <arithmetic expression>` and stop; you will get back `RESULT: <value>`. Use it for every calculation. "
            f"When you know the answer, end with exactly two lines:\nAnswer: <answer>\n{SCALE_LINE}")

def prompt_lookup(it):
    return (tat_header(it) + "You are the lookup specialist: the answer is copied from the report, not calculated. "
            + ANSWER_RULES + f"\n\nReply with exactly two lines:\nAnswer: <answer>\n{SCALE_LINE}")

def prompt_calculator(it):
    return (tat_header(it) + "You are the calculator specialist. " + PROGRAM_INSTR +
            " If the answer is a percentage, ans must be the percent number (12.5, not 0.125). "
            f"Use the report's own units. After the code block, write one line:\n{SCALE_LINE}")

def prompt_counter(it):
    return (tat_header(it) + "You are the counting specialist. List every item the question asks to count, "
            "copied from the report and separated by ' | ', on one line starting with 'Items:'. Do not write the count.")

def prompt_planner(it):
    return ("Which specialist should answer this question about a company's annual report?\n"
            "SPAN: the answer is one piece of text or one value copied from the report\n"
            "MULTI: the answer is several items copied from the report\n"
            "ARITHMETIC: the answer must be calculated from numbers in the report\n"
            "COUNT: the answer is how many items meet a condition\n\n"
            f"Question: {it['q']['question']}\n\nReply with one word: SPAN, MULTI, ARITHMETIC or COUNT.")

PLAN_TO_TYPE = {"span": "span", "multi": "multi-span", "arithmetic": "arithmetic", "count": "count"}

def user(text):
    return [{"role": "user", "content": text}]

def parse_answer(text):
    m = re.findall(r"answer\s*\**\s*[:=]\s*\**\s*(.+)", text or "", flags=re.I)
    return m[-1].strip().strip("*`").strip() if m else None

def parse_scale(text):
    m = re.findall(r"scale\s*\**\s*[:=]\s*\**\s*([a-z%/]+)", (text or "").lower())
    return SCALE_WORDS.get(m[-1], "") if m else ""

def to_prediction(ans):
    """Answer string -> TAT-QA prediction: a string, or a list of strings for several spans."""
    if ans is None:
        return None
    parts = [p.strip().strip('"').strip() for p in str(ans).split("|") if p.strip()]
    if not parts:
        return None
    return parts if len(parts) > 1 else parts[0]

def tat_score(q, pred, scale):
    m = TaTQAEmAndF1()
    m(ground_truth=q, prediction=pred if pred not in (None, [], "") else "", pred_scale=scale if scale in SCALE_WORDS.values() else "")
    em, f1, _, _ = m.get_overall_metric()
    return float(em), float(f1)

def gold_number(q):
    if q["answer_type"] in ("arithmetic", "count"):
        return float(q["answer"])
    if isinstance(q["answer"], list) and len(q["answer"]) == 1:
        return to_number(str(q["answer"][0]))
    return None

def pred_number(pred):
    if pred is None or isinstance(pred, list):
        return None
    return to_number(str(pred))

def failure_mode(q, pred, scale, em, route=None, error=None, abstained=False):
    if em == 1:
        return "correct"
    if abstained:
        return "abstained"
    if route is not None and route != GROUP[q["answer_type"]]:
        return "wrong route"
    if error:
        return "program failed"
    if pred in (None, "", []):
        return "no answer"
    g = gold_number(q)
    if g is not None:
        p = pred_number(pred)
        if p is None:
            return "text instead of number"
        if close(p, g):
            return "right number, wrong scale"
        if close(abs(p), abs(g)):
            return "sign error"
        if close(p * 100, g) or close(p / 100, g):
            return "percent vs fraction"
        return "wrong number"
    return "partial span" if tat_score(q, pred, scale)[1] > 0 else "wrong span"

def in_report(span, doc):
    norm = lambda s: " ".join(re.sub(r"[^\w.%-]", " ", str(s).lower()).split())
    return norm(span) in norm(tat_context(doc))

### Learned router (bge-m3 + logistic regression)

Trained on the TAT-QA *train* questions' answer types and tested on the sampled *dev* questions, so there is no leakage. It sees only the question, as FinSight's router would. The LLM orchestrator (M2) also sees only the question.

In [ ]:
if RUN_E4_AGENTS:
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score

    emb = SentenceTransformer(ROUTER_EMBED, device=DEVICE)
    emb.max_seq_length = 128
    if DEVICE == "cuda":
        emb.half()
    t0 = time.perf_counter()
    X_train = emb.encode([q["question"] for q in train_q], batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    router = LogisticRegression(max_iter=3000, C=4.0).fit(X_train, [q["answer_type"] for q in train_q])
    t0 = time.perf_counter()
    X_dev = emb.encode([it["q"]["question"] for it in tat_items], batch_size=128, normalize_embeddings=True)
    router_ms = (time.perf_counter() - t0) * 1000 / len(tat_items)
    router_type = list(router.predict(X_dev))
    gold_type = [it["q"]["answer_type"] for it in tat_items]
    router_acc = accuracy_score(gold_type, router_type)
    router_group_acc = np.mean([GROUP[a] == GROUP[b] for a, b in zip(router_type, gold_type)])
    print(f"Learned router: answer-type accuracy {router_acc:.3f}, macro-F1 {f1_score(gold_type, router_type, average='macro'):.3f}, "
          f"specialist accuracy {router_group_acc:.3f}, {router_ms:.1f} ms per question")
    RESULTS["E4"]["router"] = dict(model=ROUTER_EMBED, type_accuracy=router_acc, specialist_accuracy=float(router_group_acc),
                                   ms_per_question=router_ms)
    del emb, X_train
    import gc; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

### Run the arms

In [ ]:
CALC_RE = re.compile(r"^\s*`?CALC\s*:\s*(.+?)`?\s*$", re.M | re.I)

def calc(expr):
    expr = expr.strip().strip("`").replace("$", "").replace("×", "*").replace("÷", "/")
    expr = re.sub(r"(?<=\d),(?=\d{3})", "", expr)
    expr = re.sub(r"(\d+(?:\.\d+)?)\s*%", r"(\1/100)", expr)
    val, err = run_program("ans = " + expr)
    if err:
        return f"error: {err}"
    return f"{val:.6g}" if isinstance(val, (int, float)) and not isinstance(val, bool) else str(val)

def run_agent(backend, items):
    """Single agent with a calculator tool. Returns final text and all generation records per item."""
    convs = [user(prompt_agent(it)) for it in items]
    records = [[] for _ in items]
    final = [None] * len(items)
    active = list(range(len(items)))
    for step in range(AGENT_MAX_STEPS + 1):
        if not active:
            break
        if step == AGENT_MAX_STEPS:
            for i in active:
                convs[i] = convs[i] + user("Give your final answer now, as the two lines Answer and Scale.")
        outs = llm(backend, f"E4 S2 agent step {step + 1}", [convs[i] for i in active], 256)
        still = []
        for i, rec in zip(active, outs):
            records[i].append(rec)
            text = rec["text"]
            m = CALC_RE.search(text)
            answered_first = parse_answer(text) is not None and (m is None or text.lower().find("answer") < m.start())
            if m is None or answered_first or step == AGENT_MAX_STEPS:
                final[i] = text
                continue
            convs[i] = convs[i] + [{"role": "assistant", "content": text[:m.end()]}] + user(f"RESULT: {calc(m.group(1))}")
            still.append(i)
        active = still
    return final, records

def sum_cost(recs):
    return dict(calls=len(recs), in_tok=sum(r.get("in_tok") or 0 for r in recs),
                out_tok=sum(r.get("out_tok") or 0 for r in recs), sec=sum(r.get("sec") or 0 for r in recs))

def run_e4(backend, items, route_learned):
    n = len(items)
    s1 = llm(backend, "E4 S1 direct", [user(prompt_s1(it)) for it in items], 256)
    agent_text, agent_recs = run_agent(backend, items)
    lookup = llm(backend, "E4 lookup specialist", [user(prompt_lookup(it)) for it in items], 96)
    calcs = llm(backend, "E4 calculator specialist", [user(prompt_calculator(it)) for it in items], 384)
    plans = llm(backend, "E4 M2 orchestrator", [user(prompt_planner(it)) for it in items], 6)
    route_plan = []
    for p in plans:
        w = re.findall(r"span|multi|arithmetic|count", p["text"].lower())
        route_plan.append(PLAN_TO_TYPE[w[0]] if w else "span")
    gold_types = [it["q"]["answer_type"] for it in items]
    need_count = [i for i in range(n) if "count" in (route_learned[i], route_plan[i], gold_types[i])]
    counts = dict(zip(need_count, llm(backend, "E4 counter specialist", [user(prompt_counter(items[i])) for i in need_count], 160)))

    def specialist(i, answer_type):
        group = GROUP[answer_type]
        if group == "lookup":
            rec = lookup[i]
            return to_prediction(parse_answer(rec["text"])), parse_scale(rec["text"]), rec, None
        if group == "calculator":
            rec = calcs[i]
            val, err = run_program(extract_code(rec["text"]))
            if err is None and normalise_pred(val) is None:
                err = "ans is not a number"
            if err:
                return None, "", rec, err
            pred = ("yes" if val else "no") if isinstance(val, bool) else f"{float(val):.4f}"
            return pred, parse_scale(rec["text"].split("```")[-1]), rec, None
        rec = counts[i]
        m = re.findall(r"items\s*:\s*(.+)", rec["text"], flags=re.I)
        items_listed = [p for p in (m[-1].split("|") if m else []) if p.strip()]
        return (str(len(items_listed)) if items_listed else None), "", rec, None

    s1_pred = [to_prediction(parse_answer(r["text"])) for r in s1]
    s1_scale = [parse_scale(r["text"]) for r in s1]
    arms = defaultdict(list)
    for i, it in enumerate(items):
        q = it["q"]
        arms["S1 · Single LLM"].append(dict(pred=s1_pred[i], scale=s1_scale[i], cost=sum_cost([s1[i]])))
        arms["S2 · Single agent + calculator"].append(dict(pred=to_prediction(parse_answer(agent_text[i])),
                                                           scale=parse_scale(agent_text[i]), cost=sum_cost(agent_recs[i])))
        for arm, atype, extra in (("M1 · Learned router + specialists", route_learned[i], []),
                                  ("M2 · LLM orchestrator + specialists", route_plan[i], [plans[i]]),
                                  ("Oracle · Gold route + specialists", gold_types[i], [])):
            pred, scale, rec, err = specialist(i, atype)
            arms[arm].append(dict(pred=pred, scale=scale, route=GROUP[atype], error=err, cost=sum_cost(extra + [rec])))
        # M3: learned route plus a check; abstain when the check fails.
        pred, scale, rec, err = specialist(i, route_learned[i])
        group = GROUP[route_learned[i]]
        if pred is None:
            ok = False
        elif group == "lookup":
            ok = all(in_report(s, it["doc"]) for s in (pred if isinstance(pred, list) else [pred]))
        else:
            ok = answers_agree(pred, s1_pred[i] if not isinstance(s1_pred[i], list) else None)
        arms["M3 · Router + specialists + checks"].append(dict(pred=pred if ok else None, scale=scale, route=group,
                                                               error=err, abstained=(pred is not None and not ok),
                                                               cost=sum_cost([s1[i], rec])))
    for arm, rows in arms.items():
        for it, r in zip(items, rows):
            r["em"], r["f1"] = tat_score(it["q"], r["pred"], r["scale"])
            r["failure"] = failure_mode(it["q"], r["pred"], r["scale"], r["em"], r.get("route"), r.get("error"),
                                        r.get("abstained", False))
    return arms, route_plan

def report_e4(backend_name, items, arms, label=""):
    rows, base = [], np.array([r["em"] for r in arms["S2 · Single agent + calculator"]])
    for arm, rs in arms.items():
        em = np.array([r["em"] for r in rs]); f1 = np.array([r["f1"] for r in rs])
        answered = np.array([not r.get("abstained") and r["pred"] not in (None, "", []) for r in rs])
        sel = em[answered].mean() if answered.any() else float("nan")
        ci = bootstrap_ci(em)
        diff, dlo, dhi = paired_diff_ci(em, base)
        cost = pd.DataFrame([r["cost"] for r in rs]).mean()
        by_type = {t: float(em[[it["q"]["answer_type"] == t for it in items]].mean())
                   for t in ("span", "multi-span", "arithmetic", "count") if any(it["q"]["answer_type"] == t for it in items)}
        row = dict(backend=backend_name + label, arm=arm, n=len(rs), em=em.mean(), em_ci_low=ci[0], em_ci_high=ci[1],
                   f1=f1.mean(), coverage=answered.mean(), selective_em=sel, em_minus_single_agent=diff,
                   diff_ci_low=dlo, diff_ci_high=dhi, calls=cost["calls"], prompt_tokens=cost["in_tok"],
                   output_tokens=cost["out_tok"], sec=cost["sec"], **{f"em_{k}": v for k, v in by_type.items()})
        rows.append(row)
        add_summary("E4 agents", backend_name + label, arm, "exact match", em.mean(), ci, f1=f1.mean(),
                    coverage=answered.mean(), selective=sel, vs_single_agent=diff, vs_ci_low=dlo, vs_ci_high=dhi,
                    calls=cost["calls"], tokens=cost["in_tok"] + cost["out_tok"])
        print(f"{arm:38s} EM {em.mean():.3f} [{ci[0]:.3f}, {ci[1]:.3f}]  F1 {f1.mean():.3f}  cover {answered.mean():.2f}  "
              f"sel {sel:.3f}  vs S2 {diff:+.3f} [{dlo:+.3f}, {dhi:+.3f}]  calls {cost['calls']:.2f}  "
              f"tokens {cost['in_tok'] + cost['out_tok']:.0f}")
    return rows

if RUN_E4_AGENTS:
    e4_rows, e4_items_out, e4_fail = [], [], []
    for backend in BACKENDS:
        items = tat_items if backend.name == "local" else tat_items[:API_TATQA_N]
        print(f"\n=== E4 on {backend.name} ({backend.model_id}), {len(items)} questions ===")
        try:
            arms, route_plan = run_e4(backend, items, router_type[:len(items)])
        except APIStop as exc:
            print("Stopped:", exc); continue
        e4_rows += report_e4(backend.name, items, arms)
        plan_acc = np.mean([GROUP[a] == GROUP[it["q"]["answer_type"]] for a, it in zip(route_plan, items)])
        print(f"LLM orchestrator specialist accuracy: {plan_acc:.3f} (learned router: "
              f"{np.mean([GROUP[a] == GROUP[it['q']['answer_type']] for a, it in zip(router_type, items)]):.3f})")
        RESULTS["E4"][f"orchestrator_specialist_accuracy_{backend.name}"] = float(plan_acc)
        for arm, rs in arms.items():
            for f, n_ in Counter(r["failure"] for r in rs).items():
                e4_fail.append(dict(backend=backend.name, arm=arm, failure=f, count=n_, share=n_ / len(rs)))
            for it, r in zip(items, rs):
                e4_items_out.append(dict(backend=backend.name, arm=arm, uid=it["q"]["uid"], question=it["q"]["question"],
                                         answer_type=it["q"]["answer_type"], gold=it["q"]["answer"], gold_scale=it["q"]["scale"],
                                         pred=r["pred"], scale=r["scale"], em=r["em"], f1=r["f1"], failure=r["failure"],
                                         route=r.get("route"), calls=r["cost"]["calls"]))
    # Paired comparison: the local model on exactly the API's questions.
    if any(b.name == "api" for b in BACKENDS) and any(r["backend"] == "api" for r in e4_rows):
        n_api = min(API_TATQA_N, len(tat_items))
        local_items = [r for r in e4_items_out if r["backend"] == "local"]
        if local_items:
            df = pd.DataFrame(local_items)
            sub = df[df.uid.isin({it["q"]["uid"] for it in tat_items[:n_api]})]
            for arm, g in sub.groupby("arm", sort=False):
                ci = bootstrap_ci(g.em)
                e4_rows.append(dict(backend=f"local (API's {n_api} questions)", arm=arm, n=len(g), em=g.em.mean(),
                                    em_ci_low=ci[0], em_ci_high=ci[1], f1=g.f1.mean()))
                add_summary("E4 agents", f"local (API's {n_api} questions)", arm, "exact match", g.em.mean(), ci, f1=g.f1.mean())
    e4 = pd.DataFrame(e4_rows)
    e4.to_csv(os.path.join(OUT, "e4_agents.csv"), index=False)
    pd.DataFrame(e4_fail).to_csv(os.path.join(OUT, "e4_failure_modes.csv"), index=False)
    pd.DataFrame(e4_items_out).to_json(os.path.join(OUT, "e4_items.jsonl"), orient="records", lines=True, default_handler=str)
    RESULTS["E4"]["rows"] = e4.to_dict(orient="records")
    save_all()
    if e4_fail:
        print("\nFailure modes (share of questions):")
        print(pd.DataFrame(e4_fail).pivot_table(index=["backend", "arm"], columns="failure", values="share", fill_value=0).round(3).to_string())

## E5 · Memory designs on ConvFinQA

ConvFinQA conversations come from real S&P 500 earnings reports. Each conversation is a chain of questions where later ones refer back ("and what was it in 2005?", "what was the change then?"). This is the closest real public test of an assistant's working memory.

All arms answer every turn with Part 1's winning method (reason, then a Python program), and see the same report excerpt (ConvFinQA's annotated supporting facts, as in Part 1). Only the memory differs:

| Arm | Memory given at each turn |
|---|---|
| **No memory** | Only the current question |
| **Transcript** | Every earlier question with the assistant's own earlier answers (what chat apps do) |
| **Structured memory** | A short ledger of earlier results as named variables, `m1 = 60.94  # price per share in 2007`, which the program can use directly; the model writes each result's label itself (like a `memory.md` file) |

The assistant's *own* earlier answers are used, never the gold ones, so mistakes carry forward the way they would for a real user. Metrics: accuracy per turn (ConvFinQA execution accuracy with Part 1's tolerance), accuracy on follow-up turns and on turns that reuse an earlier result, whole-conversation accuracy, error carry-over, and prompt tokens per turn.

In [ ]:
MEMORY_ARMS = ["No memory", "Transcript", "Structured memory"]

def conv_context(e):
    gold = {}
    for k in ("qa", "qa_0", "qa_1"):
        gold.update((e.get(k) or {}).get("gold_inds") or {})
    if gold:
        return "\n".join(gold.values()).strip()
    return ("\n".join(e["pre_text"]) + "\n\nTable:\n" + table_to_text(e["table"]) + "\n\n" + "\n".join(e["post_text"])).strip()

def fmt_value(v):
    if v is None:
        return "unknown"
    if isinstance(v, bool):
        return "yes" if v else "no"
    try:
        return f"{float(v):.6g}"
    except (TypeError, ValueError):
        return str(v)

def uses_earlier_result(program, earlier_golds):
    if "#" in program:
        return True
    args = [parse_number(x) for x in re.findall(r"-?\d[\d.]*", program)]
    return any(a is not None and any(close(a, g, rel=1e-4, abs_tol=1e-6) for g in earlier_golds if g is not None) for a in args)

def prompt_turn(arm, e, t, history):
    q = e["annotation"]["dialogue_break"][t]
    head = f"Report excerpt:\n{conv_context(e)}\n\n"
    if arm == "No memory" or t == 0:
        return head + f"Question: {q}\n\n" + PROGRAM_INSTR
    if arm == "Transcript":
        convo = "".join(f"Q: {h['question']}\nA: {fmt_value(h['value'])}\n" for h in history)
        return head + f"Conversation so far:\n{convo}\nCurrent question: {q}\n\n" + PROGRAM_INSTR
    mem = "\n".join(f"m{j + 1} = {fmt_value(h['value']) if isinstance(h['value'], (int, float)) and not isinstance(h['value'], bool) else 'None'}"
                    f"  # {h['label']}" for j, h in enumerate(history))
    return (head + f"Memory (results of earlier questions in this conversation):\n{mem}\n\nCurrent question: {q}\n\n"
            + PROGRAM_INSTR + " You may use the memory variables m1, m2, ... directly in your code.")

LABEL_INSTR = ("\nAfter the code block, write one line 'Label: <a short self-contained description of what ans is, "
               "for example: change in revenue from 2018 to 2019>'.")

def run_e5(backend, convs):
    hist = {arm: [[] for _ in convs] for arm in MEMORY_ARMS}
    max_t = max(len(e["annotation"]["dialogue_break"]) for e in convs)
    for t in range(max_t):
        live = [c for c, e in enumerate(convs) if t < len(e["annotation"]["dialogue_break"])]
        for arm in MEMORY_ARMS:
            prompts = [prompt_turn(arm, convs[c], t, hist[arm][c]) + (LABEL_INSTR if arm == "Structured memory" else "")
                       for c in live]
            outs = llm(backend, f"E5 {arm} turn {t + 1}", [user(p) for p in prompts], 384)
            for c, p, rec in zip(live, prompts, outs):
                variables = {f"m{j + 1}": h["value"] for j, h in enumerate(hist[arm][c])
                             if isinstance(h["value"], (int, float)) and not isinstance(h["value"], bool)} \
                    if arm == "Structured memory" else None
                val, err = run_program(extract_code(rec["text"]), variables=variables)
                q = convs[c]["annotation"]["dialogue_break"][t]
                lab = re.findall(r"label\s*:\s*(.+)", rec["text"].split("```")[-1], flags=re.I)
                hist[arm][c].append(dict(question=q, value=val if err is None else None, error=err,
                                         label=(lab[-1].strip() if lab else q)[:120], rec=rec))
    return hist

def report_e5(backend_name, convs, hist, label=""):
    rows, turn_rows, items_out = [], [], []
    per_arm_correct = {}
    for arm in MEMORY_ARMS:
        recs = []
        for e, h in zip(convs, hist[arm]):
            golds = e["annotation"]["exe_ans_list"]; progs = e["annotation"]["turn_program"]
            prev_ok = None
            for t, step in enumerate(h):
                ok = finqa_correct(step["value"], golds[t])
                recs.append(dict(conv=e["id"], turn=t + 1, followup=t > 0,
                                 reuses=t > 0 and uses_earlier_result(progs[t], golds[:t]),
                                 prev_correct=prev_ok, correct=ok, in_tok=step["rec"].get("in_tok") or 0))
                items_out.append(dict(backend=backend_name, arm=arm, conv=e["id"], turn=t + 1, question=step["question"],
                                      gold=golds[t], gold_program=progs[t], pred=step["value"], error=step["error"],
                                      label=step["label"], correct=ok, output=step["rec"]["text"]))
                prev_ok = ok
        df = pd.DataFrame(recs)
        per_arm_correct[arm] = df.correct.astype(float).values
        acc = df.correct.mean(); ci = bootstrap_ci(df.correct)
        conv_acc = df.groupby("conv").correct.all().mean()
        fu = df[df.followup]; ru = df[df.reuses]
        after_wrong = fu[fu.prev_correct == False].correct.mean() if (fu.prev_correct == False).any() else float("nan")
        after_right = fu[fu.prev_correct == True].correct.mean() if (fu.prev_correct == True).any() else float("nan")
        row = dict(backend=backend_name + label, arm=arm, turns=len(df), conversations=df.conv.nunique(), accuracy=acc,
                   ci_low=ci[0], ci_high=ci[1], first_turn=df[~df.followup].correct.mean(), followup_turns=fu.correct.mean(),
                   reuse_turns=ru.correct.mean() if len(ru) else float("nan"), whole_conversation=conv_acc,
                   followup_after_wrong=after_wrong, followup_after_right=after_right, prompt_tokens=df.in_tok.mean())
        rows.append(row)
        add_summary("E5 memory", backend_name + label, arm, "turn accuracy", acc, ci, followup=fu.correct.mean(),
                    reuse=row["reuse_turns"], whole_conversation=conv_acc, prompt_tokens=df.in_tok.mean())
        for t, g in df.groupby("turn"):
            turn_rows.append(dict(backend=backend_name + label, arm=arm, turn=t, n=len(g), accuracy=g.correct.mean()))
        print(f"{arm:18s} turn acc {acc:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]  first {row['first_turn']:.3f}  "
              f"follow-up {row['followup_turns']:.3f}  reuse {row['reuse_turns']:.3f}  whole conv {conv_acc:.3f}  "
              f"tokens/turn {row['prompt_tokens']:.0f}")
    for a, b in (("Structured memory", "Transcript"), ("Structured memory", "No memory"), ("Transcript", "No memory")):
        d, lo, hi = paired_diff_ci(per_arm_correct[a], per_arm_correct[b])
        print(f"  {a} minus {b}: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")
        RESULTS["E5"].setdefault("paired", []).append(dict(backend=backend_name + label, a=a, b=b, diff=d, ci_low=lo, ci_high=hi))
    return rows, turn_rows, items_out

if RUN_E5_MEMORY:
    zpath = download("https://raw.githubusercontent.com/czyssrs/ConvFinQA/main/data.zip", "convfinqa_data.zip")
    with zipfile.ZipFile(zpath) as z:
        conv_dev = json.loads(z.read("data/dev.json"))
    conv_items = random.Random(SEED).sample(conv_dev, min(CONV_N, len(conv_dev)))
    print(f"ConvFinQA: {len(conv_items)} of {len(conv_dev)} dev conversations, "
          f"{sum(len(e['annotation']['dialogue_break']) for e in conv_items)} turns")
    e5_rows, e5_turns, e5_items = [], [], []
    for backend in BACKENDS:
        convs = conv_items if backend.name == "local" else conv_items[:API_CONV_N]
        print(f"\n=== E5 on {backend.name} ({backend.model_id}), {len(convs)} conversations ===")
        try:
            hist = run_e5(backend, convs)
        except APIStop as exc:
            print("Stopped:", exc); continue
        r, tr, io_ = report_e5(backend.name, convs, hist)
        e5_rows += r; e5_turns += tr; e5_items += io_
        if backend.name == "local" and any(b.name == "api" for b in BACKENDS):
            n_api = min(API_CONV_N, len(conv_items))
            print(f"--- local on the API's {n_api} conversations ---")
            r, tr, _ = report_e5(backend.name, conv_items[:n_api], {a: h[:n_api] for a, h in hist.items()},
                                 label=f" (API's {n_api} conversations)")
            e5_rows += r; e5_turns += tr
    e5 = pd.DataFrame(e5_rows)
    e5.to_csv(os.path.join(OUT, "e5_memory.csv"), index=False)
    pd.DataFrame(e5_turns).to_csv(os.path.join(OUT, "e5_by_turn.csv"), index=False)
    pd.DataFrame(e5_items).to_json(os.path.join(OUT, "e5_items.jsonl"), orient="records", lines=True, default_handler=str)
    RESULTS["E5"]["rows"] = e5.to_dict(orient="records")
    save_all()

## Summary

`summary.csv` holds one row per backend and arm with the primary metric and its 95% bootstrap CI. In E4, `vs_single_agent` is the paired difference from the single agent with a calculator (S2): a multi-agent arm only "wins" if that interval is above zero. In E5, compare structured memory with the transcript.

In [ ]:
summary = pd.DataFrame(SUMMARY_ROWS)
save_all()
if not summary.empty:
    print(summary.to_string(index=False))

    panels = [("E4 agents", "E4 · TAT-QA exact match"), ("E5 memory", "E5 · ConvFinQA turn accuracy")]
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    colours = {"local": "#0c6b58", "api": "#b06a00"}
    for ax, (exp, title) in zip(axes[:2], panels):
        part = summary[(summary.experiment == exp) & summary.backend.isin(["local", "api"])]
        if part.empty:
            ax.set_axis_off(); continue
        labels = list(dict.fromkeys(part.method))
        y = np.arange(len(labels)); h = 0.38
        for k, (bk, g) in enumerate(part.groupby("backend", sort=False)):
            g = g.set_index("method").reindex(labels)
            ax.barh(y + (k - 0.5) * h, g.value, height=h, color=colours.get(bk, "#555"), label=bk)
            ax.errorbar(g.value, y + (k - 0.5) * h, xerr=[g.value - g.ci_low, g.ci_high - g.value],
                        fmt="none", ecolor="#333", capsize=2)
        ax.set_yticks(y, labels, fontsize=8); ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_title(title); ax.legend()
    ax = axes[2]
    if RUN_E5_MEMORY and e5_turns:
        bt = pd.DataFrame(e5_turns)
        bt = bt[(bt.backend == "local") & (bt.n >= 10)]
        for arm, g in bt.groupby("arm", sort=False):
            ax.plot(g.turn, g.accuracy, marker="o", label=arm)
        ax.set_xlabel("turn in conversation"); ax.set_ylabel("accuracy"); ax.set_ylim(0, 1)
        ax.set_title("E5 · accuracy by turn (local)"); ax.legend()
    else:
        ax.set_axis_off()
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "summary.png"), dpi=150); plt.show()
print("Saved to", OUT)